# 07 · SR 리포트 — **메인 표 + 그림**

01~06 에서 나온 반복 eval 결과를 **전 모델** 한 표로. 논문 Table 1.
- 수치 = 150k ckpt × 5 rep × 4 seed = **20 run** 의 **mean ± std** + pooled Wilson 95% CI.
- 그림 = 모델별 막대(±std) + **개별 run 을 점으로** (산포가 보이게).
- 결과 없는 모델은 자동으로 빠짐 → 일부만 돌린 상태에서도 그대로 실행 가능.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion'. cf.SHORT_SIM('transfer') / cf.SUPPORT_SIM 로 바꿔 같은 리포트를 낼 수 있음
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
GPUS  = cf.v23.available_gpus()   # 이 노드에 실제 보이는 GPU
NGPU  = len(GPUS)                 # 4개면 4잡씩 청크로 (하드코딩 X)
TAGS = cf.FINAL_TAGS       # baseline 6 + ours (있는 것만 표에 나옴)
REPS = list(range(cf.EVAL_REPEATS))
N_EP = cf.EVAL_N_EP

out = cf.OUTPUT_BASE / 'main_report'
out.mkdir(parents=True, exist_ok=True)
print('task:', TASK, '| seeds:', SEEDS, '| reps:', REPS)

## 표 → `main_report/sr_150k_reps.csv`

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP,
                   csv_path=out / 'sr_150k_reps.csv')

## 그림 — 모델별 SR (점 = 개별 run)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

fig, ax = plt.subplots(figsize=(11, 5))
for i, r in enumerate(rows):
    t = r['tag']
    agg = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    ax.bar(i, agg['mean'], yerr=agg['std'], color=cf.COLOR.get(t, '#333'),
           alpha=0.85, capsize=5, width=0.62)
    ax.scatter([i] * len(agg['all']), agg['all'], s=12, color='k', alpha=0.35, zorder=3)
ax.set_xticks(range(len(rows)))
ax.set_xticklabels([r['model'] for r in rows], rotation=20, ha='right')
ax.set_ylabel('Success rate (%)')
ax.set_title(f"{TASK} @ {cf.CKPT_STEP // 1000}k — {len(REPS)} reps x {len(SEEDS)} seeds",
             fontweight='bold')
fig.savefig(out / 'sr_150k_reps.png')
fig.savefig(out / 'sr_150k_reps.pdf')
plt.show()
print('저장:', out / 'sr_150k_reps.png')

## (참고) SR vs step 곡선 — **학습중 eval 을 켠 경우에만**
기본 설정은 학습중 eval **OFF**(`cf.v23.EVAL_FREQ = 0`) → 이 셀은 "데이터 없음"을 출력한다.
논문 수치는 위의 150k 반복 eval 로 낸다. 수렴 곡선이 필요하면 `common_final.py` 에서
`v23.EVAL_FREQ = 10_000` 으로 되돌리고 다시 학습할 것.

In [ ]:
import numpy as np
curves_found = {t: [c for c in (cf.eval_curve(t, s, TASK) for s in SEEDS) if c] for t in TAGS}
curves_found = {t: v for t, v in curves_found.items() if v}
if not curves_found:
    print('학습중 eval 데이터 없음 (EVAL_FREQ=0 로 껐음) — 정상.')
    print('논문 수치는 위의 150k 반복 eval 표/그림을 쓸 것.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for t, curves in curves_found.items():
        steps = sorted(set().union(*[set(c) for c in curves]))
        mean = [np.mean([c[st] for c in curves if st in c]) for st in steps]
        ax.plot([s / 1000 for s in steps], mean, '-o', ms=4,
                color=cf.COLOR.get(t, '#333'), label=cf.FINAL_LABELS.get(t, t))
    ax.axvline(cf.CKPT_STEP / 1000, ls='--', c='r', alpha=0.6, label='eval ckpt (150k)')
    ax.set(xlabel='step (k)', ylabel='SR (%)', title='SR vs step (학습중 eval, seed 평균)')
    ax.legend(fontsize=10)
    fig.savefig(out / 'sr_curve.png')
    plt.show()